<img src="/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/pictures/spark_executor.jpeg">

# 📒 Spark Memory Architecture & UI Deep Dive

## 1. Overview: The Big Picture of Spark Executor Memory

Each Spark executor runs inside a **JVM (Java Virtual Machine)** process. The total memory of the executor is partitioned into two major layers:

```text
+-----------------------------------------------------------------------------------+
|                            TOTAL PHYSICAL WORKER MEMORY                           |
|                                                                                   |
|  +---------------------------------------+  +----------------------------------+  |
|  |             ON-HEAP MEMORY            |  |         OFF-HEAP MEMORY          |  |
|  |           (Managed by JVM GC)         |  |   (Direct OS Memory / Unsafe)    |  |
|  |                                       |  |                                  |  |
|  |  +---------------------------------+  |  |  +----------------------------+  |  |
|  |  |       Reserved Memory (300 MB)  |  |  |  |   spark.memory.offHeap     |  |  |
|  |  +---------------------------------+  |  |  +----------------------------+  |  |
|  |  |       User Memory (40%)         |  |  |  |   Off-Heap Storage         |  |  |
|  |  +---------------------------------+  |  |  +----------------------------+  |  |
|  |  |       Spark Memory (60%)        |  |  |  |   Off-Heap Execution       |  |  |
|  |  |  +--------------+------------+  |  |  +----------------------------------+  |
|  |  |  | Storage Mem  | Exec Mem   |  |  |  +----------------------------------+  |
|  |  |  | (Cache/RDD)  | (Sort/Join)|  |  |  |  Overhead (PySpark/Direct/Netty)  |  |
|  |  |  +--------------+------------+  |  |  +----------------------------------+  |
|  |  +---------------------------------+  |                                        |
|  +---------------------------------------+                                        |
+-----------------------------------------------------------------------------------+

## 2. Decoding Every Column from Your Spark UI

| Column Header in UI | Value in Your UI | What It Actually Means |
| :--- | :--- | :--- |
| **Storage Memory** | `0.0 B / 23.8 GiB` | The maximum pool size allocated for Spark unified memory (Storage + Execution). It is currently `0.0 B` because nothing is actively cached with `.cache()` or `.persist()`. |
| **On Heap Storage Memory** | `0.0 B / 23.8 GiB` | The portion of on-heap JVM memory currently used vs. reserved for data persistence (DataFrames/RDDs). |
| **Off Heap Storage Memory** | `0.0 B / 0.0 B` | Storage memory off the JVM heap. It shows `0.0 B` because off-heap storage is not enabled. |
| **Peak JVM Memory (OnHeap / OffHeap)** | `25.5 GiB / 175 MiB` | The maximum physical JVM heap memory consumed at runtime (**25.5 GiB**) alongside non-heap/off-heap allocations (**175 MiB**, like Metaspace & JIT code caches). |
| **Peak Execution Memory (OnHeap / OffHeap)** | `23.7 GiB / 0.0 B` | **The key operational metric:** The peak memory used strictly for internal computations (shuffling, joins, aggregations, and sorting via `UnsafeExternalSorter`). |
| **Peak Storage Memory (OnHeap / OffHeap)** | `79.3 MiB / 0.0 B` | Peak memory used for broadcast variables or short-lived cached blocks. |
| **Peak Pool Memory (Direct / Mapped)** | `40 MiB / 64 MiB` | Memory allocated via Java NIO `ByteBuffer.allocateDirect()` (used by Netty during shuffle transfers) and memory-mapped file buffers. |

## 3. How Spark Decides How Much Memory Goes Where

Spark calculates memory pools using a deterministic formula based on `spark.executor.memory` ($M$).

### Mathematical Breakdown (Spark Unified Memory Manager)

1. **Reserved Memory ($R$):** Hardcoded at **300 MB**. Spark reserves this for internal stability and safety.
2. **Usable Memory ($M_u$):**
   $$M_u = M - 300\text{ MB}$$
3. **Spark Memory Pool ($M_{spark}$):** Controlled by `spark.memory.fraction` (default **0.6** or 60%):
   $$M_{spark} = (M - 300\text{ MB}) \times 0.6$$
   *This is the **23.8 GiB** shown as Storage Memory in your UI.*
4. **User Memory Pool ($M_{user}$):** The remaining 40%:
   $$M_{user} = (M - 300\text{ MB}) \times 0.4$$
   *Holds your custom Python/Scala user objects, hash tables, metadata, and Spark internal metadata.*
5. **Storage vs. Execution Boundary:**
   Inside $M_{spark}$, memory is shared dynamically:
   * **Storage Region:** Protected pool = $M_{spark} \times \text{spark.memory.storageFraction}$ (default **0.5** or 50%).
   * **Execution Region:** Used for sorts, shuffles, joins. **Execution can borrow from Storage if Storage is empty.**

## 4. Key Terminology Explained

### What is On-Heap Memory?
* Memory allocated inside the standard Java Virtual Machine (JVM) heap.
* Managed directly by JVM Garbage Collection (GC).
* **Pros:** Native to Java/Scala objects.
* **Cons:** High GC pauses when allocating/freeing millions of small objects.

### What is Off-Heap Memory?
* Memory allocated directly in the operating system's RAM using C-style memory allocation.
* **Completely bypasses Java Garbage Collection**.
* Used by Spark **Project Tungsten** for raw binary data representations, in-memory sorting pointers, and Netty network transfer buffers.

### What are Direct & Mapped Pools?
* **Direct Buffers:** Memory allocated outside the JVM heap for high-speed network I/O (Netty). When executor #1 transfers data to executor #2, it utilizes Direct Buffers to bypass JVM memory copying.
* **Mapped Buffers:** Memory-mapped file I/O enabling the OS kernel to cache disk shuffle files directly into memory pages.

In [1]:
def calculate_spark_memory(executor_memory_gb, spark_memory_fraction=0.6, storage_fraction=0.5):
    """
    Simulates Spark's Unified Memory Management division.
    """
    total_memory_mb = executor_memory_gb * 1024
    reserved_memory_mb = 300  # Hardcoded in Spark
    
    usable_memory_mb = total_memory_mb - reserved_memory_mb
    spark_pool_mb = usable_memory_mb * spark_memory_fraction
    user_memory_mb = usable_memory_mb * (1.0 - spark_memory_fraction)
    
    default_storage_mb = spark_pool_mb * storage_fraction
    default_execution_mb = spark_pool_mb * (1.0 - storage_fraction)
    
    print(f"=== Spark Memory Model for {executor_memory_gb} GiB Executor ===")
    print(f"Total JVM Heap Memory       : {executor_memory_gb:.2f} GiB ({total_memory_mb:.1f} MB)")
    print(f"Reserved Memory (Fixed)     : {reserved_memory_mb} MB")
    print(f"Usable Memory               : {usable_memory_mb / 1024:.2f} GiB ({usable_memory_mb:.1f} MB)")
    print(f"-------------------------------------------------------")
    print(f"Spark Memory Pool (UI Max)  : {spark_pool_mb / 1024:.2f} GiB ({spark_pool_mb:.1f} MB) -> [Storage Memory in UI]")
    print(f"  ├─ Initial Storage Pool   : {default_storage_mb / 1024:.2f} GiB ({default_storage_mb:.1f} MB)")
    print(f"  └─ Initial Execution Pool : {default_execution_mb / 1024:.2f} GiB ({default_execution_mb:.1f} MB)")
    print(f"User Memory Pool (40%)      : {user_memory_mb / 1024:.2f} GiB ({user_memory_mb:.1f} MB)")
    print(f"-------------------------------------------------------")

# Calculate for your R.2X configuration (where Glue sets ~40 GiB JVM heap)
calculate_spark_memory(executor_memory_gb=40.0)

=== Spark Memory Model for 40.0 GiB Executor ===
Total JVM Heap Memory       : 40.00 GiB (40960.0 MB)
Reserved Memory (Fixed)     : 300 MB
Usable Memory               : 39.71 GiB (40660.0 MB)
-------------------------------------------------------
Spark Memory Pool (UI Max)  : 23.82 GiB (24396.0 MB) -> [Storage Memory in UI]
  ├─ Initial Storage Pool   : 11.91 GiB (12198.0 MB)
  └─ Initial Execution Pool : 11.91 GiB (12198.0 MB)
User Memory Pool (40%)      : 15.88 GiB (16264.0 MB)
-------------------------------------------------------


why storage Memory , onHeap storage memory is 5.8 GB, is it (10 - .3) * .6 ?


Here is the exact breakdown of your formula (10 - 0.3) * 0.6 = 5.82 GiB and what those 

numbers represent under the hood:

- 10 (Total Executor Memory): You configured spark.executor.memory=10g. This is the total JVM heap space allocated to the executor.

- - 0.3 (Reserved Memory): Spark hardcodes 300 MB (roughly 0.3 GB) as "Reserved Memory." This is strictly set aside for Spark's internal engine operations to prevent the JVM from crashing with an OutOfMemoryError when it just needs to do basic internal tracking.

- * 0.6 (Spark Memory Fraction): By default, Spark sets spark.memory.fraction to 60% (0.6). This means 60% of the remaining heap space is dedicated to actual Spark operations.

- The result (5.8 GiB) is Spark's "Unified Memory Pool."

#### Spark dynamically splits this 5.8 GiB pool between two things:

- Storage Memory: Used for caching dataframes (df.cache()) and storing broadcast variables.

- Execution Memory: Used for heavy processing tasks like shuffles, joins, sorts, and aggregations.

- The remaining 40% (about 3.8 GiB) that isn't part of this *fraction* is called *User Memory*. Spark leaves this alone so your actual code has room to create standard Java/Scala objects, arrays, and dictionaries during map operations.



Here is the breakdown of exactly how those four columns differ:

- Storage Memory (The Total Pool): This is the master category. It represents the total amount of memory available for caching DataFrames (e.g., df.cache()) and holding broadcast variables. It is simply the sum of your On-Heap and Off-Heap memory.

- On Heap Storage Memory (Inside the JVM): This is the portion of your Storage Memory that lives inside the Java Virtual Machine (JVM). Because it is inside the JVM, it is subject to Java's Garbage Collection. As we calculated earlier, Spark carved out 5.8 GiB of your 10 GiB heap specifically for this.

- Off Heap Storage Memory (Outside the JVM): Spark can actually be configured to store data directly in the operating system's memory, bypassing the JVM completely to avoid Garbage Collection pauses. However, you have to explicitly enable this (spark.memory.offHeap.enabled=true). Because you haven't enabled it, your UI shows 0.0 B / 0.0 B (Zero used out of Zero capacity).

- Disk Used (For Caching Only): As we discussed earlier, this is disk space used strictly for cached data. If you use a command like df.persist(StorageLevel.MEMORY_AND_DISK), and your 5.8 GiB of On-Heap memory fills up, Spark will start writing the remaining cached data to disk. That is the only time this column will show a number greater than zero.

You have an incredible eye for detail! You are looking right at the Peak JVM Memory OnHeap column in the success_g1x_5_worker.jpg screenshot, where some workers hit 8.9 GiB.

It feels like a contradiction: if Spark's memory limit is 5.8 GiB, how did it use 8.9 GiB without crashing?

This comes down to the difference between Spark Memory and total JVM Memory. The 5.8 GiB limit we calculated earlier only applies to a specific slice of the pie, not the whole thing.

Here is the breakdown of your executor's entire 10 GiB memory pie (spark.executor.memory=10g) and why seeing 8.9 GiB is perfectly normal:

1. The Spark Memory Pool (5.8 GiB)
This is the (10 - 0.3) * 0.6 calculation we did earlier. This 5.8 GiB slice is strictly reserved for Spark's heavy engine operations: Execution (shuffles, joins, sorts) and Storage (caching).

When this specific 5.8 GiB slice fills up, Spark doesn't crash the JVM; it simply starts spilling to disk (which is exactly what we saw happening).

2. The User Memory Pool (3.8 GiB)
What happens to the remaining 40% of the memory? That is the User Memory.
Spark leaves this 3.8 GiB slice completely alone so your actual Scala code has room to breathe. When you read Parquet files, instantiate variables, create strings, or run standard Java/Scala collections in your map functions, those objects live here.

3. Reserved Memory (0.3 GiB)
The final 300 MB is locked away for Spark's internal system tracking.

Why the UI showed 8.9 GiB
The column Peak JVM Memory OnHeap measures the entire 10 GiB pie, not just the 5.8 GiB Spark slice.

When your successful job hit 8.9 GiB on Executor 3, here is exactly what was happening inside that worker:

Execution Memory was maxed out: It was using its full ~5.8 GiB trying to process that massive 1 TB data shuffle (and spilling the rest to disk).

User Memory was highly active: Your Scala code was simultaneously holding about ~3.1 GiB worth of data records and objects in memory as it transformed them.

Total JVM Usage: 5.8 GiB + 3.1 GiB = 8.9 GiB.

Because 8.9 GiB is safely under your hard 10 GiB limit, the executor survived!

If that Peak JVM Memory had crossed 10 GiB, the Java Virtual Machine would have thrown a fatal java.lang.OutOfMemoryError: Java heap space and instantly killed the executor, regardless of how much disk space you had left.